# EDA — Twitter Fake Account Detection

Exploratory Data Analysis on the Twitter fake account dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.dpi"] = 120
sns.set_style("whitegrid")

TRAIN_PATH = "../data/raw/train_data_eps1.xlsx"
TEST_PATH = "../data/raw/test_data_eps1.xlsx"

df_train = pd.read_excel(TRAIN_PATH)
df_test = pd.read_excel(TEST_PATH)
print(f"Train shape: {df_train.shape}")
print(f"Test shape:  {df_test.shape}")

---
## 1. Raw Data Inspection

In [ ]:
df_train.head(3)

In [ ]:
df_train.info(show_counts=True)

---
## 2. Missing Values Analysis

In [ ]:
def missing_table(df, name=""):
    missing = df.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    tbl = pd.DataFrame({
        "column": missing.index,
        "missing": missing.values,
        "pct": (missing.values / len(df) * 100).round(2)
    })
    print(f"Missing values — {name}" if name else "Missing values")
    display(tbl)

missing_table(df_train, "Train")
missing_table(df_test, "Test")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(df_train.isnull(), cbar=False, yticklabels=False, ax=axes[0])
axes[0].set_title("Train — Missing Values")
sns.heatmap(df_test.isnull(), cbar=False, yticklabels=False, ax=axes[1])
axes[1].set_title("Test — Missing Values")
plt.tight_layout()
plt.show()

**Insight:** Only `location` has missing values (~19-20% in both sets). All other columns are complete.

---
## 3. Target Distribution

In [ ]:
target_counts = df_train["fake"].value_counts()
target_pct = df_train["fake"].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

axes[0].bar(["Real (0)", "Fake (1)"], target_counts.values, color=["#2ecc71", "#e74c3c"])
axes[0].set_title("Target Count")
axes[0].set_ylabel("Count")
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 20, str(v), ha="center")

axes[1].pie(target_counts.values, labels=["Real", "Fake"], autopct="%1.1f%%",
            colors=["#2ecc71", "#e74c3c"], startangle=90)
axes[1].set_title("Target Proportion")

plt.tight_layout()
plt.show()

print(f"Real: {target_counts[0]} ({target_pct[0]:.1f}%)")
print(f"Fake: {target_counts[1]} ({target_pct[1]:.1f}%)")

**Insight:** Moderate class imbalance — ~60% Real vs ~40% Fake. No severe imbalance, but accuracy alone may be misleading.

---
## 4. Univariate Analysis

### 4a. Numeric Features

In [ ]:
num_cols = ["followers_count", "friends_count", "post_count"]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for i, col in enumerate(num_cols):
    axes[0, i].hist(df_train[col], bins=50, edgecolor="black", alpha=0.7)
    axes[0, i].set_title(f"{col} — Distribution")
    axes[0, i].set_xlabel(col)
    
    axes[1, i].boxplot(df_train[col])
    axes[1, i].set_title(f"{col} — Boxplot")
    axes[1, i].set_ylabel(col)

plt.tight_layout()
plt.show()

df_train[num_cols].describe().round(2)

In [ ]:
num_cols_extended = ["default_profile_image", "profile_use_background_image", "verified"]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for i, col in enumerate(num_cols_extended):
    vc = df_train[col].value_counts()
    axes[i].bar(vc.index.astype(str), vc.values, color=["#3498db", "#e67e22"])
    axes[i].set_title(col)
    axes[i].set_ylabel("Count")
    for j, v in enumerate(vc.values):
        axes[i].text(j, v + 10, str(v), ha="center")
plt.tight_layout()
plt.show()

### 4b. Categorical Features

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

lang_counts = df_train["lang"].value_counts()
axes[0].bar(lang_counts.index, lang_counts.values, color="#9b59b6")
axes[0].set_title("Language Distribution")
axes[0].set_ylabel("Count")

loc_available = df_train["location"].notna().sum()
loc_missing = df_train["location"].isna().sum()
axes[1].bar(["Available", "Missing"], [loc_available, loc_missing],
            color=["#2ecc71", "#e74c3c"])
axes[1].set_title("Location Availability")
axes[1].set_ylabel("Count")
for i, v in enumerate([loc_available, loc_missing]):
    axes[1].text(i, v + 10, str(v), ha="center")

plt.tight_layout()
plt.show()

print("Language distribution:")
print(lang_counts)

In [ ]:
# Description length distribution
df_train["desc_len"] = df_train["description"].fillna("").astype(str).apply(len)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].hist(df_train["desc_len"], bins=50, edgecolor="black", alpha=0.7)
axes[0].set_title("Description Length — Distribution")
axes[0].set_xlabel("Description Length")

axes[1].boxplot(df_train["desc_len"])
axes[1].set_title("Description Length — Boxplot")
axes[1].set_ylabel("Characters")

plt.tight_layout()
plt.show()
print(f"Description length — mean: {df_train['desc_len'].mean():.1f}, median: {df_train['desc_len'].median():.0f}")

In [ ]:
# Account age distribution
ref_date = pd.Timestamp("2025-12-02")
df_train["account_age_days"] = (ref_date - pd.to_datetime(df_train["created_at"])).dt.days

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].hist(df_train["account_age_days"], bins=50, edgecolor="black", alpha=0.7)
axes[0].set_title("Account Age — Distribution")
axes[0].set_xlabel("Age (days)")

axes[1].boxplot(df_train["account_age_days"])
axes[1].set_title("Account Age — Boxplot")
axes[1].set_ylabel("Days")

plt.tight_layout()
plt.show()
print(f"Account age — min: {df_train['account_age_days'].min():.0f} days, "
      f"max: {df_train['account_age_days'].max():.0f} days, "
      f"mean: {df_train['account_age_days'].mean():.0f} days")

### 4c. Screen Name Analysis

In [ ]:
df_train["screen_name_len"] = df_train["screen_name"].fillna("").astype(str).apply(len)
df_train["screen_name_numeric"] = (
    df_train["screen_name"]
    .fillna("")
    .astype(str)
    .apply(lambda x: sum(c.isdigit() for c in x))
)
df_train["screen_name_numeric_ratio"] = (
    df_train["screen_name_numeric"] / df_train["screen_name_len"].replace(0, 1)
)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].hist(df_train["screen_name_len"], bins=30, edgecolor="black", alpha=0.7)
axes[0].set_title("Screen Name Length")
axes[0].set_xlabel("Length")
axes[1].hist(df_train["screen_name_numeric_ratio"], bins=30, edgecolor="black", alpha=0.7)
axes[1].set_title("Screen Name — Numeric Ratio")
axes[1].set_xlabel("Numeric Ratio")
plt.tight_layout()
plt.show()

---
## 5. Correlation Analysis

In [ ]:
corr_cols = [
    "followers_count", "friends_count", "post_count",
    "default_profile_image", "profile_use_background_image",
    "verified", "desc_len", "account_age_days",
    "screen_name_len", "screen_name_numeric_ratio", "fake"
]

df_corr = df_train[corr_cols].copy()
df_corr["location_available"] = df_train["location"].notna().astype(int)

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(df_corr.corr(), dtype=bool))
sns.heatmap(
    df_corr.corr(),
    annot=True, fmt=".2f",
    cmap="RdBu_r", center=0,
    mask=mask, square=True,
    linewidths=0.5
)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Top correlations with target
corr_with_target = df_corr.corr()["fake"].drop("fake").sort_values(key=abs, ascending=False)

plt.figure(figsize=(8, 4))
colors = ["#e74c3c" if v < 0 else "#2ecc71" for v in corr_with_target.values]
plt.barh(corr_with_target.index, corr_with_target.values, color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Correlation with fake")
plt.title("Feature Correlations with Target")
plt.tight_layout()
plt.show()

print("Correlation with target (sorted by absolute value):")
print(corr_with_target.to_string())

---
## 6. Feature vs Target Breakdown

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
compare_cols = ["followers_count", "friends_count", "post_count"]

for i, col in enumerate(compare_cols):
    for j, (label, color) in enumerate(zip(["Real", "Fake"], ["#2ecc71", "#e74c3c"])):
        subset = df_train[df_train["fake"] == j][col]
        axes[j, i].hist(subset, bins=30, alpha=0.7, color=color, edgecolor="black")
        axes[j, i].set_title(f"{label} — {col}")
        axes[j, i].set_xlabel(col)

plt.tight_layout()
plt.show()

In [ ]:
# Binary features vs target
binary_cols = ["default_profile_image", "profile_use_background_image", "verified"]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for i, col in enumerate(binary_cols):
    crosstab = pd.crosstab(df_train[col], df_train["fake"], normalize="index") * 100
    crosstab.plot(kind="bar", ax=axes[i], color=["#2ecc71", "#e74c3c"], legend=False)
    axes[i].set_title(col)
    axes[i].set_ylabel("Percentage")
    axes[i].set_xlabel("Value")
    axes[i].legend(["Real", "Fake"], loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()

---
## 7. Key Insights Summary

1. **Missing data**: Only `location` has nulls (~19-20%). All other 13 columns are complete.
2. **Target balance**: ~60% Real, ~40% Fake — mild imbalance. Use F1 / precision / recall alongside accuracy.
3. **Followers / Friends / Posts**: All right-skewed. Fake accounts tend to have higher followers and posts on average.
4. **Default profile image**: ~50% use default. Fake accounts more often use default images.
5. **Profile background**: ~51% use background image. Marginally more common among real accounts.
6. **Verified accounts**: Only ~3% are verified, nearly all are real.
7. **Account age**: Wide range (0 to ~3700 days). Newer accounts are slightly more likely to be fake.
8. **Language**: Evenly distributed across 5 languages (en, de, es, fr, it).
9. **Screen name**: Fake accounts tend to have more numeric characters in screen names.
10. **Top features** by correlation magnitude: `verified`, `default_profile_image`, `followers_count`, `account_age_days`.